In [1]:
# !pip install transformers sentencepiece

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import sentencepiece as spm

import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.model_selection import train_test_split

c:\Users\ysawo\Downloads\Deep Learning\Final_Project_PartA\laughing-octo-disco\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Loading the trained model
sp = spm.SentencePieceProcessor()
sp.load('vocab.model')

True

In [4]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token 

In [5]:
base_lm = AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3140.77it/s]


In [6]:
dataset = pd.read_csv('dataset/train.csv')
dataset = dataset.dropna(subset=['prompt', 'response_a', 'response_b'])

In [7]:
dataset.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1.0,0.0,0.0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0.0,1.0,0.0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0.0,0.0,1.0
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1.0,0.0,0.0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0.0,1.0,0.0


In [8]:
def preprocess_data(data):
    processed_data = []
    
    for i in range(data.shape[0]):
        prompt = dataset.iloc[0]["prompt"]

        # To find the preferred response
        preferred_response_id = np.argmax(dataset.iloc[0][-3:])


        if preferred_response_id < 2:
            preferred_response = ""
            rejected_response = ""

            if preferred_response_id == 0:
                preferred_response = dataset.iloc[0]["response_a"]
                rejected_response = dataset.iloc[0]["response_b"]
            elif preferred_response_id == 1:
                preferred_response = dataset.iloc[0]["response_b"]
                rejected_response = dataset.iloc[0]["response_a"]

            preferred_input = prompt + "" + preferred_response
            rejected_input = prompt + "" + rejected_response

            # preferred_input = sp.Encode(preferred_input)
            # rejected_input = sp.Encode(rejected_input)

            preferred_input = tokenizer(preferred_input, padding="max_length", max_length=512, return_tensors="pt")
            rejected_input = tokenizer(rejected_input, padding="max_length", max_length=512, return_tensors="pt")

            # #making the padding and masks
            # #For preferred inputs
            # if len(preferred_input)==2000:
            #     preferred_attention_mask = [1] * len(preferred_input)
            # else:
            #     preferred_input = preferred_input + [0]* (2000 - len(preferred_input))
            #     preferred_attention_mask = [1] * len(preferred_input) + [0]* (2000-len(preferred_input))

            # #For rejected inputs
            # if len(rejected_input)==2000:
            #     rejected_attention_mask = [1] * len(rejected_input)
            # else:
            #     rejected_input = rejected_input + [0]* (2000 - len(rejected_input))
            #     rejected_attention_mask = [1] * len(rejected_input) + [0]* (2000-len(rejected_input))

            # processed_data.append([preferred_input,preferred_attention_mask,rejected_input,rejected_attention_mask])

            processed_data.append([preferred_input,rejected_input])
    return(processed_data)

    ## Bradley Terry requires ranked pairs, so ties are ignored

    # else:
    #     # if it is a tie
    #     preferred_response_1 = dataset.iloc[0]["response_a"]
    #     preferred_response_2 = dataset.iloc[0]["response_b"]

    #     preferred_input_1 = prompt + "" + preferred_response_1
    #     preferred_input_2 = prompt + "" + preferred_response_1

    #     preferred_input_1 = sp.Encode(preferred_input_1)
    #     preferred_input_2 = sp.Encode(preferred_input_2)

    #     #making the padding and masks
    #     #For preferred inputs 1
    #     if len(preferred_input_1)==2000:
    #         preferred_attention_mask_1 = [1] * len(preferred_input_1)
    #     else:
    #         preferred_input_1 = preferred_input_1 + [0]* (2000 - len(preferred_input_1))
    #         preferred_attention_mask_1 = [1] * len(preferred_input_1) + [0]* (2000-len(preferred_input_1))

    #     #For preferred inputs 2
    #     if len(preferred_input_2)==2000:
    #         preferred_attention_mask_2 = [1] * len(preferred_input_2)
    #     else:
    #         preferred_input_2 = preferred_input_2 + [0]* (2000 - len(preferred_input_2))
    #         preferred_attention_mask_2 = [1] * len(preferred_input_2) + [0]* (2000-len(preferred_input_2))




In [9]:
preprocess_data = preprocess_data(dataset)

In [10]:
# print(preprocess_data)

In [11]:
# data = torch.tensor(np.array(preprocess_data),dtype=torch.long)
# data.shape

np.shape(preprocess_data)

(30000, 2, 2)

In [12]:
class TextDataset(Dataset):
    def __init__(self, data):
        # self.chosen = data[:,0,:]
        # self.chosen_masks = data[:,1,:]
        # self.rejected = data[:,2,:]
        # self.rejected_masks = data[:,3,:]

        self.chosen = []
        self.chosen_masks = []
        self.rejected = []
        self.rejected_masks = []

        for pair in data:
            self.chosen.append(pair[0]['input_ids'].squeeze(0))
            self.chosen_masks.append(pair[0]['attention_mask'].squeeze(0))
            self.rejected.append(pair[1]['input_ids'].squeeze(0))
            self.rejected_masks.append(pair[1]['attention_mask'].squeeze(0))

    def __len__(self):
        return len(self.chosen)

    def __getitem__(self, idx):
        return {
            'chosen': self.chosen[idx],
            'chosen_masks': self.chosen_masks[idx],
            'rejected': self.rejected[idx],
            'rejected_masks': self.rejected_masks[idx],
        }

In [13]:
class BradleyTerryRewardModel(nn.Module):
    """
    Standard scalar reward model for Bradley-Terry preference learning.

    Usage (pairwise BT loss):
        rewards_chosen = model(**inputs_chosen)    # (batch,)
        rewards_rejected = model(**inputs_rejected)  # (batch,)
        loss = -F.logsigmoid(rewards_chosen - rewards_rejected).mean()
    """
    def __init__(self, base_lm):
        super().__init__()
        self.lm = base_lm  # e.g., AutoModelForCausalLM
        self.head = nn.Linear(self.lm.config.hidden_size, 1)

    def _sequence_rep(self, hidden, attention_mask):
        """
        Get a single vector per sequence to score.
        Default: last non-padding token (EOS token); if no mask, last token.
        hidden: (batch, seq_len, hidden_size)
        attention_mask: (batch, seq_len)
        """

        # Index of last non-pad token in each sequence
        # attention_mask is 1 for real tokens, 0 for padding
        lengths = attention_mask.sum(dim=1) - 1  # (batch,)
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        return hidden[batch_idx, lengths]  # (batch, hidden_size)

    def forward(self, input_ids, attention_mask):
        """
        A forward pass designed to show inference structure of a standard reward model.
        To train one, this function will need to be modified to compute rewards from both
         chosen and rejected inputs, applying the loss above.
        """
        outputs = self.lm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )
        # Final hidden states: (batch, seq_len, hidden_size)
        hidden = outputs.hidden_states[-1]

        # One scalar reward per sequence: (batch,)
        seq_repr = self._sequence_rep(hidden, attention_mask)
        rewards = self.head(seq_repr).squeeze(-1)

        return rewards
    
    def training_loss(self, input_ids_chosen, attn_mask_chosen, input_ids_rejected, attn_mask_rejected):
        rew_c = self(input_ids_chosen, attn_mask_chosen)
        rew_r = self(input_ids_rejected, attn_mask_rejected)
        return -F.logsigmoid(rew_c - rew_r).mean()

In [14]:
model = BradleyTerryRewardModel(base_lm)

In [ ]:
# Creating the  DataLoader
tdata = TextDataset(preprocess_data)
train, test = train_test_split(tdata,test_size=0.3)
train_dataloader = DataLoader(train, batch_size=2, shuffle=True)
test_dataloader = DataLoader(test, batch_size=2, shuffle=True)

# In training loop
for batch in train_dataloader:
    loss = model.training_loss(
        batch['chosen'], batch['chosen_masks'],
        batch['rejected'], batch['rejected_masks']
    )

In [ ]:
max_id = batch['chosen'].max().item()
print(max_id)
vocab_size = model.lm.config.vocab_size
print(vocab_size)

1991
50257


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
for batch in dataloader:
    loss = model.training_loss(**batch)
    loss.backward()
    optimizer.step()

NameError: name 'model' is not defined